In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
# from lightgbm import LGBMRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import r2_score, mean_squared_error



In [ ]:
df = pd.read_csv("data/data_join.csv")

print(df.shape)
df.head()

In [ ]:
df["DATE"] = pd.to_datetime(df["DATE"], format="%d-%m-%Y")

df["month"] = df["DATE"].dt.month
df["year"] = df["DATE"].dt.year
df["dayofyear"] = df["DATE"].dt.dayofyear

In [ ]:
features = [
    "LATITUDE",
    "LONGITUDE",
    "PET",
    "NIR",
    "GREEN",
    "SWIR16",
    "SWIR22",
    "NDMI",
    "MNDWI",
    "month",
    "dayofyear"
]

targets = [
    "TOTALAL_KALINITY",
    "ELECTRICAL_CONDUCTANCE",
    "DISSOLVED_REACTIVE_PHOSPHORUS"
]


X = df[features]
y = df[targets]

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

dummy = DummyRegressor(strategy="mean")
dummy.fit(X_train, y_train)

pred = dummy.predict(X_test)


r2 = r2_score(y_test, pred)

print("Dummy R2:", r2)
print("Dummy RMSE:", np.sqrt(mean_squared_error(y_test, pred)))

In [ ]:
for i in range(0, 3):
    r2 = r2_score(y_test, pred)

    print("Dummy R2:", r2)

In [ ]:
from sklearn.model_selection import KFold


kf = KFold(n_splits=5, shuffle=True, random_state=42)

r2_scores = []

for train_idx, test_idx in kf.split(X):

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    dummy = DummyRegressor(strategy="mean")
    dummy.fit(X_train, y_train)

    pred = dummy.predict(X_test)

    r2 = r2_score(y_test, pred)

    r2_scores.append(r2)

print("CV R2:", np.mean(r2_scores))

In [ ]:

model = LGBMRegressor(
    n_estimators=200,
    random_state=42,
    verbose=-1
)

for i in targets:
    y = df[i]

    scores = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring="r2"
    )

    print("Model CV R2:", scores.mean())